In [1]:
from ultralytics import YOLO
import yaml
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
import os, random, json
from PIL import Image, ImageOps
from collections import Counter
from scipy.stats import mode
from skimage import measure

In [2]:
TRAINED_MODEL_PATH = '/workspace/user3/runs/detect/yolo26s_custom_bbox_model/weights/best.pt'

# 베스트 모델 불러오기.
loaded_model = YOLO(TRAINED_MODEL_PATH)

In [3]:
fix_label_load = pd.read_csv('/workspace/데이터셋/[노션] 3차 프로젝트(4조)-오분류 감정 라벨링.csv')
fix_label = fix_label_load.iloc[:,:5]
change_label = fix_label[~(fix_label['이전 감정']==fix_label['정정 감정'])]
box_error_img = fix_label[fix_label['BOX 이상']=='Yes']

In [4]:
ADRESS = '/workspace/데이터셋'

img_adress = f'{ADRESS}/img'
label_adress = f'{ADRESS}/label'
segmentation_adress = f'{ADRESS}/segmentation'

In [5]:
DATA_TYPES_TO_PROCESS = ['train', 'val']
EMOTIONS = ['anger', 'happy', 'sadness', 'panic']

In [6]:
def load_emotion_data_files(emotion, data_type):
    """
    주어진 감정과 데이터 유형에 대한 json 라벨과 npz 세그먼트를 불러오는 함수.
    (json_data, npz_data)가 디폴트 반환값이지만 데이터를 찾을 수 없는 경우 (None, None)을 반환
    """
    json_file_path = os.path.join(label_adress, data_type, f'{data_type}_{emotion}.json')
    npz_file_path = os.path.join(segmentation_adress, data_type, f'{data_type}_{emotion}.npz')

    current_json_data = None
    current_npz_data = None

    # json 데이터 로드
    if os.path.exists(json_file_path):
        try:
            with open(json_file_path, 'r', encoding='cp949') as f:
                current_json_data = json.load(f)
            print(f"Loaded JSON for {emotion} ({data_type}) from {json_file_path}")
        except Exception as e:
            print(f"Error loading JSON for {emotion} ({data_type}) from {json_file_path}: {e}")
    else:
        print(f"JSON file not found: {json_file_path}")

    # npz 데이터 로드
    if os.path.exists(npz_file_path):
        try:
            current_npz_data = np.load(npz_file_path)
            print(f"Loaded NPZ for {emotion} ({data_type}) from {npz_file_path}")
        except Exception as e:
            print(f"Error loading NPZ for {emotion} ({data_type}) from {npz_file_path}: {e}")
    else:
        print(f"NPZ file not found: {npz_file_path}")

    return current_json_data, current_npz_data

In [7]:
def combined_data_iterator():
    """
    이미지, 라벨, 세그먼트 데이터를 매칭시켜주는 함수
    EMOTIONS, DATA_TYPES_TO_PROCESS 변수로 지정된 데이터 유형과 감정에 대해서만 조회.
    """
    for data_type in DATA_TYPES_TO_PROCESS:
        for emotion in EMOTIONS:
            json_labels, npz_segments = load_emotion_data_files(emotion, data_type)

            # 라벨, 세그먼트 중에 하나라도 누락되어있을 경우 스킵.
            if json_labels is None or npz_segments is None:
                print(f"Skipping {emotion} {data_type} due to missing JSON or NPZ data.")
                if npz_segments is not None: npz_segments.close()
                continue

            # img 데이터 로드
            img_emotion_dir = os.path.join(img_adress, data_type, emotion)

            # 이미지가 누락되어있을 경우 스킵.
            if not os.path.exists(img_emotion_dir):
                print(f"Image directory not found: {img_emotion_dir}. Skipping this emotion for {data_type}.")
                if npz_segments is not None: npz_segments.close()
                continue

            for json_entry in json_labels:
                filename = json_entry.get('filename')
                if not filename: continue # 이미지 파일 이름이 없으면 스킵.
                
                # 재분류된 이미지 라벨 정정
                current_emotion = emotion
                if change_label.파일명.str.contains(filename.split('.')[0]).any():
                    current_emotion = change_label.loc[change_label.파일명.str.contains(filename.split('.')[0]),'정정 감정'].iloc[0]

                boxes_err = False
                if box_error_img.파일명.str.contains(filename.split('.')[0]).any(): boxes_err = True

                img_full_path = os.path.join(img_emotion_dir, filename)
                segmentation_mask = npz_segments.get(filename)

                # 파일 매칭. 이미지 + 라벨 + 세그먼트
                if os.path.exists(img_full_path) and segmentation_mask is not None:
                    entry_data = {
                        'filename': filename,
                        'data_type': data_type,
                        'gender': json_entry.get('gender'),
                        'age': json_entry.get('age'),
                        'img_path': img_full_path,
                        'faceExp': current_emotion,
                        'segmentation_mask': segmentation_mask,
                      # 'isProf': json_entry.get('isProf'),
                      # 'faceExp_uploader': json_entry.get('faceExp_uploader'),
                      # 'bg_uploader': json_entry.get('bg_uploader'),
                    }

                    min_x_vals, min_y_vals, max_x_vals, max_y_vals = [], [], [], []
                    for annotator in ['A', 'B', 'C']:
                        annot_key = f'annot_{annotator}'
                        if annot_key in json_entry and isinstance(json_entry[annot_key], dict):
                            boxes = json_entry[annot_key].get('boxes')
                            if boxes and all(k in boxes for k in ['minX', 'minY', 'maxX', 'maxY']):
                                min_x_vals.append(boxes['minX'])
                                min_y_vals.append(boxes['minY'])
                                max_x_vals.append(boxes['maxX'])
                                max_y_vals.append(boxes['maxY'])

                    avg_min_x, avg_min_y, avg_max_x, avg_max_y = None, None, None, None
                    if min_x_vals:
                        avg_min_x = np.mean(min_x_vals).item()
                        avg_min_y = np.mean(min_y_vals).item()
                        avg_max_x = np.mean(max_x_vals).item()
                        avg_max_y = np.mean(max_y_vals).item()
                        entry_data['boxes'] = {'maxX': avg_min_x, 'maxY': avg_min_y, 'minX': avg_max_x, 'minY': avg_max_y}
                        entry_data['boxes_err'] = boxes_err

                    yield entry_data

            if npz_segments is not None: npz_segments.close()

In [9]:
_original_data_types_to_process = DATA_TYPES_TO_PROCESS
_original_emotions = EMOTIONS
DATA_TYPES_TO_PROCESS = ['val']
EMOTIONS = ['anger']

temp_val_iterator = combined_data_iterator()

In [38]:
temp_val_iterator1 = next(temp_val_iterator)

In [ ]:
temp_val_iterator

{'filename': '7uji1c8ab7778bc286c3cd90077847f9a906f00b210838ba57877a29abd60ffix.jpg',
 'data_type': 'val',
 'gender': '남',
 'age': 40,
 'img_path': '/workspace/데이터셋/img/val/anger/7uji1c8ab7778bc286c3cd90077847f9a906f00b210838ba57877a29abd60ffix.jpg',
 'faceExp': 'anger',
 'segmentation_mask': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 4, 4, 4],
        [0, 0, 0, ..., 4, 4, 4],
        [0, 0, 0, ..., 4, 4, 4]], shape=(2208, 2944), dtype=uint8),
 'boxes': {'maxX': 1622.2562555288696,
  'maxY': 333.6447799999999,
  'minX': 2333.5914752708136,
  'minY': 1321.8902999999998},
 'boxes_err': False}

In [39]:
TEST_IMAGE_PATH = temp_val_iterator1['img_path']

# 원래대로 복원
DATA_TYPES_TO_PROCESS = _original_data_types_to_process
EMOTIONS = _original_emotions

results = loaded_model.predict(source=TEST_IMAGE_PATH, save=True, conf=0.1, iou=0.7)
print(temp_val_iterator1['faceExp'])


image 1/1 /workspace/데이터셋/img/val/anger/tzg8926848c43051598de35ece59588857ce02af1699541b88484a1bede3ev7xl.jpg: 800x1024 1 blank, 1 anger, 103.7ms
Speed: 5.5ms preprocess, 103.7ms inference, 4.7ms postprocess per image at shape (1, 3, 800, 1024)
Results saved to /workspace/user3/runs/detect/predict5
anger


In [18]:
blank = []
anger = []
happy = []
sadness = []
panic = []

for r in results:
    # 1. 탐지된 객체가 있는지 확인
    if r.boxes is not None:
        for box in r.boxes:
            # 현재 1순위 클래스 인덱스와 확률
            cls_id = int(box.cls[0])
            conf_1st = float(box.conf[0])
            label_1st = r.names[cls_id]
            if label_1st == ""



            print(f"Detected: {label_1st} with confidence {conf_1st:.4f}")

SyntaxError: expected ':' (324077772.py, line 15)

In [40]:
for r in results:
    if r.boxes is not None:
        r.boxes

In [42]:
r.boxes[0]

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([1.], device='cuda:0')
conf: tensor([0.5705], device='cuda:0')
data: tensor([[1.1828e+03, 8.0752e+00, 2.0931e+03, 1.2306e+03, 5.7053e-01, 1.0000e+00]], device='cuda:0')
id: None
is_track: False
orig_shape: (2320, 3088)
shape: torch.Size([1, 6])
xywh: tensor([[1637.9922,  619.3170,  910.2905, 1222.4836]], device='cuda:0')
xywhn: tensor([[0.5304, 0.2669, 0.2948, 0.5269]], device='cuda:0')
xyxy: tensor([[1182.8469,    8.0752, 2093.1375, 1230.5588]], device='cuda:0')
xyxyn: tensor([[0.3830, 0.0035, 0.6778, 0.5304]], device='cuda:0')

In [41]:
r.boxes[1]

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([0.], device='cuda:0')
conf: tensor([0.2514], device='cuda:0')
data: tensor([[1.1850e+03, 4.0293e+00, 2.0946e+03, 1.2271e+03, 2.5144e-01, 0.0000e+00]], device='cuda:0')
id: None
is_track: False
orig_shape: (2320, 3088)
shape: torch.Size([1, 6])
xywh: tensor([[1639.7881,  615.5854,  909.6051, 1223.1123]], device='cuda:0')
xywhn: tensor([[0.5310, 0.2653, 0.2946, 0.5272]], device='cuda:0')
xyxy: tensor([[1184.9855,    4.0293, 2094.5906, 1227.1416]], device='cuda:0')
xyxyn: tensor([[0.3837, 0.0017, 0.6783, 0.5289]], device='cuda:0')

In [43]:
r.boxes[2]

IndexError: index 2 is out of bounds for dimension 0 with size 2

In [ ]:
results

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'blank', 1: 'anger', 2: 'happy', 3: 'sadness', 4: 'panic'}
 obb: None
 orig_img: array([[[ 0,  4,  0],
         [ 4,  8,  3],
         [ 7,  8,  6],
         ...,
         [22,  9,  7],
         [19,  6,  4],
         [15,  2,  0]],
 
        [[ 2,  6,  1],
         [ 6, 10,  5],
         [ 8,  9,  7],
         ...,
         [31, 18, 16],
         [28, 15, 13],
         [19,  6,  4]],
 
        [[ 7,  8,  6],
         [10, 11,  9],
         [ 9, 10,  8],
         ...,
         [26, 13, 11],
         [25, 12, 10],
         [21,  8,  6]],
 
        ...,
 
        [[29, 20, 23],
         [36, 27, 30],
         [41, 32, 35],
         ...,
         [24, 16, 17],
         [28, 20, 21],
         [37, 29, 30]],
 
        [[55, 46, 49],
         [51, 42, 45],
         [44, 35, 38],
         ...,
         [27, 19, 20],
         [29, 21, 22],
    

- checkpoint: 엥 디텍트 모델도 왜 확률분포를 가져올 수 없는 걸까...

In [ ]:
TEST_IMAGE_PATH = '/workspace/user3/2026-04-20_4.03.51.png'
results = loaded_model.predict(source=TEST_IMAGE_PATH, save=True, conf=0.1, iou=0.7)


image 1/1 /workspace/user3/2026-04-20_4.03.51.png: 512x1024 (no detections), 10.6ms
Speed: 3.8ms preprocess, 10.6ms inference, 0.3ms postprocess per image at shape (1, 3, 512, 1024)
Results saved to /workspace/user3/runs/detect/predict2


In [8]:
VIDEO_LIST = [
    '/workspace/user4/소풍_관객반응.mp4',
]

for path in VIDEO_LIST:
    if os.path.exists(path):
        results = loaded_model.predict(source=path, stream=True, save=True, conf=0.25, iou=0.7)
        for r in results: # stream=True 시 필수 실행
            pass



video 1/1 (frame 1/1112) /workspace/user4/소풍_관객반응.mp4: 576x1024 (no detections), 122.2ms
video 1/1 (frame 2/1112) /workspace/user4/소풍_관객반응.mp4: 576x1024 (no detections), 11.3ms
video 1/1 (frame 3/1112) /workspace/user4/소풍_관객반응.mp4: 576x1024 (no detections), 11.3ms
video 1/1 (frame 4/1112) /workspace/user4/소풍_관객반응.mp4: 576x1024 (no detections), 11.2ms
video 1/1 (frame 5/1112) /workspace/user4/소풍_관객반응.mp4: 576x1024 (no detections), 11.5ms
video 1/1 (frame 6/1112) /workspace/user4/소풍_관객반응.mp4: 576x1024 (no detections), 11.3ms
video 1/1 (frame 7/1112) /workspace/user4/소풍_관객반응.mp4: 576x1024 (no detections), 11.1ms
video 1/1 (frame 8/1112) /workspace/user4/소풍_관객반응.mp4: 576x1024 (no detections), 11.1ms
video 1/1 (frame 9/1112) /workspace/user4/소풍_관객반응.mp4: 576x1024 (no detections), 10.8ms
video 1/1 (frame 10/1112) /workspace/user4/소풍_관객반응.mp4: 576x1024 (no detections), 11.1ms
video 1